# NS-Forest workflow

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
CODE_PATH = "/mnt/session_data/NSForest" # location of NSForest folder
sys.path.insert(0, os.path.abspath(CODE_PATH))
from nsforest import ns, nsforesting, utils, NSFOREST_VERSION


In [ ]:
pd.set_option('display.max_rows', None)

## 0. Set up

In [ ]:
## set up
organ = "pancreas" #<---
author = "Tritshcler" #<---
year = "2022" #<---
output_folder = "outputs_" + organ + "_" + author + "_" + year + "/" 

cluster_header = "louvain_anno_fine" #<---
outputfilename_suffix = cluster_header
outputfilename_prefix = cluster_header 

## 1. Data

#### [need to filter normal cells -- Anne]

In [ ]:
data_folder = "/home/jovyan/session_data/" #<---

In [ ]:
adata = sc.read_h5ad(data_folder + "pancreas_Tritschler_2022_00d88707-e33a-4c2a-821a-cdc32a98d050.h5ad") #<---

In [ ]:
adata

## 2. Clusters

#### number of clusters

In [ ]:
## number of clusters
n_clusters = adata.obs[cluster_header].nunique()
n_clusters

#### dendrogram (run this to automatically create the output folder)

In [ ]:
## auto-adjust figsize
fig_width = int(n_clusters/5)
fig_height = max([2, int(max([len(z) for z in adata.obs[cluster_header].unique()]) / 30) + 1])

In [ ]:
## dendrogram and save svg
ns.pp.dendrogram(adata, cluster_header, figsize = (fig_width, fig_height), tl_kwargs = {'optimal_ordering': True},
                 save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_suffix)

#### cluster sizes

In [ ]:
## cluster sizes
df_cluster_sizes = pd.DataFrame(adata.obs[cluster_header].value_counts())
df_cluster_sizes

In [ ]:
## save
df_cluster_sizes.to_csv(output_folder + outputfilename_prefix + "_cluster_sizes.csv")

#### cluster order

In [ ]:
cluster_order = [x.strip() for x in adata.uns["dendrogram_" + cluster_header]['categories_ordered']]

In [ ]:
## save
pd.DataFrame({'cluster_order': cluster_order}).to_csv(output_folder + outputfilename_prefix + "_cluster_order.csv", index=False)

#### summary statistics of data (normal cells)

In [ ]:
df_normal = pd.DataFrame({'n_obs': [adata.n_obs], 'n_vars': [adata.n_vars], 'n_clusters': [n_clusters]})
df_normal

In [ ]:
## save
df_normal.to_csv(output_folder + outputfilename_prefix + "_summary_normal.csv", index=False)

## 3. NS-Forest 

#### prep

In [ ]:
## make a copy b/c the median step will only keep the positive genes
## keep the original data for plotting
adata_prep = adata.copy()

In [ ]:
## get medians
adata_prep = ns.pp.prep_medians(adata_prep, cluster_header)

In [ ]:
## get binary scores
adata_prep = ns.pp.prep_binary_scores(adata_prep, cluster_header)

In [ ]:
## check medians
df_medians = adata_prep.varm['medians_' + cluster_header]
print(df_medians.shape)
df_medians.head()

In [ ]:
## check binary scores
df_binary_scores = adata_prep.varm['binary_scores_' + cluster_header]
print(df_binary_scores.shape)
df_binary_scores.head()

In [ ]:
## save csv and pkl
df_medians.to_csv(output_folder + outputfilename_prefix + "_medians.csv")
df_medians.to_pickle(output_folder + outputfilename_prefix + "_medians.pkl")

df_binary_scores.to_csv(output_folder + outputfilename_prefix + "_binary_scores.csv")
df_binary_scores.to_pickle(output_folder + outputfilename_prefix + "_binary_scores.pkl")

#### histograms of  non-zero  values [TO-DO: nice to have them as functions -- Beverly]

In [ ]:
non_zero_medians = df_medians[df_medians != 0].stack().values

In [ ]:
plt.hist(non_zero_medians, bins=100)
plt.title("Non-zero medians")

plt.savefig(output_folder + "hist_nonzero_medians_" + outputfilename_suffix + ".svg")
plt.show()

In [ ]:
non_zero_binary_scores = df_binary_scores[df_binary_scores != 0].stack().values

In [ ]:
plt.hist(non_zero_binary_scores, bins=100)
plt.title("Non-zero binary scores")

plt.savefig(output_folder + "hist_nonzero_binary_scores_" + outputfilename_suffix + ".svg")
plt.show()

#### run `NSForest()`

In [ ]:
results = nsforesting.NSForest(adata_prep, cluster_header, save = True, save_supplementary = True,
                               output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
results

## 4. Plotting

#### **load NS-Forest results** (copy set up and load pkl)

In [ ]:
# ## set up
# organ = "kidney" #<---
# author = "Lake" #<---
# year = "2023" #<---
# output_folder = "outputs_" + organ + "_" + author + "_" + year + "/" #e.g., "outputs_kidney_Lake_2023/"

# cluster_header = "subclass.full" #<---
# outputfilename_suffix = cluster_header
# outputfilename_prefix = cluster_header 

# ## load NS-Forest results
# results = pd.read_pickle(output_folder + cluster_header + "_results.pkl")

In [ ]:
## set results to plot
results_to_plot = results

#### boxplots

In [ ]:
ns.pl.boxplot(results_to_plot, "f_score", save = "html", output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
ns.pl.boxplot(results_to_plot, "precision", save = "html", output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
ns.pl.boxplot(results_to_plot, "recall", save = "html", output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
ns.pl.boxplot(results_to_plot, "onTarget", save = "html", output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

#### scatter plots w.r.t. cluster size

In [ ]:
ns.pl.scatter_w_clusterSize(results, "f_score", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
ns.pl.scatter_w_clusterSize(results, "precision", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
ns.pl.scatter_w_clusterSize(results, "recall", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

In [ ]:
ns.pl.scatter_w_clusterSize(results, "onTarget", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)

#### scanpy plots [TO-DO: map ENSGs to gene symbols using Ray's repo -- Anne & Ray]

In [ ]:
## make the markers dictionary for plotting
markers_dict = dict(zip(results_to_plot["clusterName"], results_to_plot["NSForest_markers"]))

In [ ]:
ns.pl.dotplot(adata, markers_dict, cluster_header, dendrogram = True, use_raw = False,
              save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_prefix)

In [ ]:
ns.pl.dotplot(adata, markers_dict, cluster_header, dendrogram = True, use_raw = False, standard_scale = 'var', 
              save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_prefix + "_scaled")

In [ ]:
ns.pl.stackedviolin(adata, markers_dict, cluster_header, dendrogram = True, use_raw = False,
                    save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_prefix)

In [ ]:
ns.pl.stackedviolin(adata, markers_dict, cluster_header, dendrogram = True, use_raw = False, standard_scale = 'var',
                    save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_prefix + "_scaled")

In [ ]:
ns.pl.matrixplot(adata, markers_dict, cluster_header, dendrogram = True, use_raw = False,
                 save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_prefix)

In [ ]:
ns.pl.matrixplot(adata, markers_dict, cluster_header, dendrogram = True, use_raw = False, standard_scale = 'var',
                 save = "svg", output_folder = output_folder, outputfilename_suffix = outputfilename_prefix + "_scaled")

# Save

#### count saved items in the output folder

In [ ]:
from pathlib import Path
folder_path = Path(output_folder) 
item_count = len(list(folder_path.iterdir()))
print(f"Total items in the output folder: {item_count}")

#### saving html report

In [ ]:
## for interactive plot
import plotly
plotly.offline.init_notebook_mode()

In [ ]:
## save html report
!jupyter nbconvert --to html pancreas_Tritschler_NS-Forest_workflow.ipynb #<---